In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Define the directory path
directory_path = '../data/raw/'

# Load train raw data for app, sms, user, and voc
# Read the CSV files directly
app_data = pd.read_csv(f'{directory_path}train/train_app.csv')
sms_data = pd.read_csv(f'{directory_path}train/train_sms.csv')
user_data = pd.read_csv(f'{directory_path}train/train_user.csv')
voc_data = pd.read_csv(f'{directory_path}train/train_voc.csv')


/var/folders/3c/t2jl0gwn1p731dmykyzxz5cw0000gn/T/ipykernel_4280/4034817127.py:13: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  voc_data = pd.read_csv(f'{directory_path}train/train_voc.csv')


### App Dataset

In [17]:
# Ensure 'month_id' is unique per dataset instance to avoid recalculating
num_months = app_data['month_id'].nunique()

# Grouping and aggregating data
app_data_grouped = app_data.groupby('phone_no_m').agg(
    app_usage_count=('busi_name', 'nunique'),
    flow_mean=('flow', 'mean'),
    flow_median=('flow', 'median'),
    flow_min=('flow', 'min'),
    flow_max=('flow', 'max'),
    flow_var=('flow', 'var'),
    flow_sum=('flow', 'sum'),
    months_count=('month_id', 'nunique')
).reset_index()

app_data_grouped['flow_month'] = app_data_grouped['flow_sum'] / app_data_grouped['months_count'].replace(0, 1)

### SMS Dataset

In [18]:
# SMS data preprocessing
sms_data['request_datetime'] = pd.to_datetime(sms_data['request_datetime'])
sms_data['hour'] = sms_data['request_datetime'].dt.hour
sms_data['date'] = sms_data['request_datetime'].dt.date
sms_data['day'] = sms_data['request_datetime'].dt.day
sms_data['day_name'] = sms_data['request_datetime'].dt.day_name()

sms_data_grouped = sms_data.groupby('phone_no_m').agg(
    sms_count=('phone_no_m', 'size'),
    sms_nunique_contact=('opposite_no_m', 'nunique'),
    sms_date_nunique=('date', 'nunique'),  # Unique days when SMS was sent
    sms_hour_mode=('hour', lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    sms_hour_mode_count=('hour', lambda x: x.value_counts().iloc[0] if not x.value_counts().empty else np.nan),
    sms_hour_nunique=('hour', 'nunique'),
    sms_day_mode=('day', lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    sms_day_mode_count=('day', lambda x: x.value_counts().iloc[0] if not x.value_counts().empty else np.nan),
    sms_day_nunique=('day', 'nunique'),
    sms_dayname_mode_count=('day_name', lambda x: x.value_counts().iloc[0] if not x.value_counts().empty else np.nan),
    sms_dayname_nunique=('day_name', 'nunique')
).reset_index()

sms_data_grouped['sms_rate'] = sms_data_grouped['sms_count']/sms_data_grouped['sms_nunique_contact']
sms_data_grouped['sms_contacts_proportion'] = sms_data_grouped['sms_nunique_contact']/sms_data_grouped['sms_count']

# Group by phone_no_m and calltype_id, count occurrences, and pivot into separate columns
grouped_data = sms_data.groupby(['phone_no_m', 'calltype_id']).size().unstack(fill_value=0)
for call_type in [1, 2, 3]:
    if call_type not in grouped_data.columns:
        grouped_data[call_type] = 0  # Add missing call types with zero count
        
sms_data_grouped = sms_data_grouped.merge(grouped_data[[1, 2]], on='phone_no_m', how='left').fillna(0)
sms_data_grouped.rename(columns={1: 'sms_calltype1_count', 2: 'sms_calltype2_count'}, inplace=True)
sms_data_grouped['sms_calltype2_proportion'] = sms_data_grouped['sms_calltype2_count']/sms_data_grouped['sms_count']

### VOC Dataset

In [19]:
# VOC data preprocessing
voc_data['start_datetime'] = pd.to_datetime(voc_data['start_datetime'])
voc_data['hour'] = voc_data['start_datetime'].dt.hour
voc_data['day'] = voc_data['start_datetime'].dt.day
voc_data['day_name'] = voc_data['start_datetime'].dt.day_name()

voc_data_grouped = voc_data.groupby('phone_no_m').agg(
    imei_count=('imei_m', lambda x: len(set(x))),
    imei_list=('imei_m', lambda x: list(set(x))),  # Store unique IMEI as a list
    call_count=('phone_no_m', 'size'),
    call_unique=('opposite_no_m', 'nunique'),
    call_city_unique=('city_name', 'nunique'),
    call_county_unique=('county_name', 'nunique'),
    call_count_mean=('opposite_no_m', lambda x: x.value_counts().mean() if len(x) > 0 else 0),
    call_count_median=('opposite_no_m', lambda x: x.value_counts().median() if len(x) > 0 else 0),    
    call_count_max=('opposite_no_m', lambda x: x.value_counts().max() if len(x) > 0 else 0),
    call_dur_mean=('call_dur', 'mean'),
    call_dur_median=('call_dur', 'median'),
    call_dur_min=('call_dur', 'min'),
    call_dur_max=('call_dur', 'max'),
    voc_hour_mode=('hour', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 0),
    voc_hour_mode_count=('hour', lambda x: x.value_counts().get(x.mode().iloc[0], 0) if len(x.mode()) > 0 else 0),
    voc_hour_unique=('hour', 'nunique'),
    voc_day_mode=('day', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 0),
    voc_day_mode_count=('day', lambda x: x.value_counts().get(x.mode().iloc[0], 0) if len(x.mode()) > 0 else 0),
    voc_date_unique=('day', 'nunique'),
    voc_dayname_mode_count=('day_name', lambda x: x.value_counts().get(x.mode().iloc[0], 0) if len(x.mode()) > 0 else 0),
    voc_dayname_unique=('day_name', 'nunique'),
    calltypeid_unique=('calltype_id', 'nunique')
).reset_index()

# Group by phone_no_m and calltype_id, count occurrences, and pivot into separate columns
grouped_data = voc_data.groupby(['phone_no_m', 'calltype_id']).size().unstack(fill_value=0)
# Ensure all call types exist to avoid KeyErrors
for call_type in [1, 2, 3]:
    if call_type not in grouped_data.columns:
        grouped_data[call_type] = 0  # Add missing call types with zero count

voc_data_grouped = voc_data_grouped.merge(grouped_data[[1, 2, 3]], on='phone_no_m', how='left').fillna(0)
voc_data_grouped.rename(columns={1: 'voc_calltype1_count', 2: 'voc_calltype2_count', 3: 'voccalltype3_count'}, inplace=True)
voc_data_grouped['voc_calltype1_proportion'] = voc_data_grouped['voc_calltype1_count']/voc_data_grouped['call_count']


# Filter outgoing calls (calltype_id = 1)
outgoing_calls = voc_data[voc_data['calltype_id'] == 1]

# Calculate unique cities and counties for outgoing calls
call_outgoing_city_unique = outgoing_calls.groupby('phone_no_m')['city_name'].nunique().reset_index(name='call_outgoing_city_unique')
call_outgoing_county_unique = outgoing_calls.groupby('phone_no_m')['county_name'].nunique().reset_index(name='call_outgoing_county_unique')

# Merge the results back into voc_data_grouped
voc_data_grouped = voc_data_grouped.merge(call_outgoing_city_unique, on='phone_no_m', how='left')
voc_data_grouped = voc_data_grouped.merge(call_outgoing_county_unique, on='phone_no_m', how='left')

# Calculate the median of the sum of call durations for each phone_no_m with unique opposite_no_m
phone2oppo_sum = voc_data.groupby(['phone_no_m', 'opposite_no_m'])['call_dur'].sum().reset_index()
phone2oppo_sum_median = phone2oppo_sum.groupby('phone_no_m')['call_dur'].median().reset_index(name='call_duration_per_contact_median')

# Calculate the max of the sum of call durations for each phone_no_m with unique opposite_no_m
phone2oppo_sum_max = phone2oppo_sum.groupby('phone_no_m')['call_dur'].max().reset_index(name='call_duration_per_contact_max')

# Merge the median and max values back into voc_data_grouped
voc_data_grouped = voc_data_grouped.merge(phone2oppo_sum_median, on='phone_no_m', how='left')
voc_data_grouped = voc_data_grouped.merge(phone2oppo_sum_max, on='phone_no_m', how='left')

# Calculate the mean total duration of calls over the unique number of contacts
voc_data_grouped['call_duration_per_contact_mean'] = voc_data_grouped['call_dur_mean'] * voc_data_grouped['call_count'] / voc_data_grouped['call_unique']

### User Dataset

In [20]:
# Identify ARPU columns
arpu_columns = [col for col in user_data.columns if col.startswith('arpu_')]

# Fill NaN values with 0 for ARPU columns
user_data[arpu_columns] = user_data[arpu_columns].fillna(0)

# Compute ARPU statistics, treating 0s as NaNs to exclude them from calculations
user_data['arpu_mean'] = user_data[arpu_columns].replace(0, np.nan).mean(axis=1)
user_data['arpu_var'] = user_data[arpu_columns].replace(0, np.nan).var(axis=1)
user_data['arpu_max'] = user_data[arpu_columns].replace(0, np.nan).max(axis=1)
user_data['arpu_min'] = user_data[arpu_columns].replace(0, np.nan).min(axis=1)
user_data['arpu_median'] = user_data[arpu_columns].replace(0, np.nan).median(axis=1)
user_data['arpu_sum'] = user_data[arpu_columns].replace(0, np.nan).sum(axis=1)
user_data['arpu_skew'] = user_data[arpu_columns].replace(0, np.nan).skew(axis=1)
user_data['arpu_sem'] = user_data[arpu_columns].replace(0, np.nan).sem(axis=1)

# Select only relevant columns
columns_to_keep = [
    'phone_no_m', 'arpu_mean', 'arpu_var', 'arpu_max', 'arpu_min', 'arpu_median',
    'arpu_sum', 'arpu_skew', 'arpu_sem', 'idcard_cnt', 'label'
]

user_data_grouped = user_data[columns_to_keep].copy()

In [21]:
# Combine the user, app, and sms data
df = user_data_grouped.merge(app_data_grouped, on='phone_no_m', how='left')
df = df.merge(voc_data_grouped, on='phone_no_m', how='left')
df = df.merge(sms_data_grouped, on='phone_no_m', how='left')

In [22]:
# Save the processed data
df.to_csv('../data/processed/processed_dataset.csv', index=False)